# 从 compressed IR 到二维结构：test10.mps 探索指南

本示例用 MIPFrame 的通用查询逐步验证模型结构。我们先观察 family，再检查真实行、
变量关联和精确系数，最后评估一个代入想法的非零项代价。它不运行求解器，也不修改模型。

运行前请在仓库环境中安装 mipframe，并确保 data/test10.mps 可用（约 28 MB）。
Notebook 从仓库根目录或 examples 目录启动均可。首次读取会计算 compressed IR，
需要一些时间。文中的 F0、V2 等标签只对这个模型和默认 template 配置有效。

In [1]:
from pathlib import Path
import mipframe as mf
from mipframe import col

source = next(
    (root / "data" / "test10.mps"
     for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data" / "test10.mps").is_file()),
    None,
)
if source is None:
    raise FileNotFoundError("找不到 data/test10.mps；请从仓库目录启动 Notebook，或手动设置 source")

m = mf.read(source)
m

MIPModel("test10")
  sense        : min
  variables    : 190,402
  constraints  : 2,303
  nonzeros     : 761,756
  density      : 0.00174
  var types    : B 190,201 | C 201
  objective nz : 1

## 1. 先读模型地图

默认 compressed IR 展示模型规模、变量角色、约束 family 和逐层细化结果。
这里约有 19 万个变量、2303 条约束；F0 与 F1 各有 950 行，F2 有
200 行，V2 有 190000 个变量。IR 的结构是规范化多重集合，不能代替原始公式。

In [ ]:
m.ir

## 2. 把 family ID 锚定到原始成员

先看行名、变量名和非零项数。Frame 的 head() 只显示成员片段；一个小 family
在默认显示上限内会完整列出。索引显示的是原始模型 ID。

按原始 ID 查成员用 `.by_id(id)`，按名称用 `.by_name(name)`；
`.iloc[位置]` 和 `.iloc[起点:终点]` 始终按当前选区的位置访问。
`m.rows["F0"]` 则属于独立的 family 标签命名空间。已有单行对象时直接用 `row.vars`。

In [3]:
F0 = m.rows.family("F0@2")
F1 = m.rows.family("F1@2")
F2 = m.rows.family("F2@2")
V1 = m.vars.family("V1@1")
V2 = m.vars.family("V2@1")

display(F0.head(), F1.head(), F2.head())
display(V1.head(), V2.head())

<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R0001  R0001  0  inf  201
R0002  R0002  0  inf  201
R0003  R0003  0  inf  201
R0004  R0004  0  inf  201
R0005  R0005  0  inf  201
<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R0951  R0951  1  1  200
R0952  R0952  1  1  200
R0953  R0953  1  1  200
R0954  R0954  1  1  200
R0955  R0955  1  1  200
<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R2101  R2101  0  0  951
R2102  R2102  0  0  951
R2103  R2103  0  0  951
R2104  R2104  0  0  951
R2105  R2105  0  0  951
<VarFrame key=('var',) · total not computed>
var  name  kind  lb  ub  obj  type  nnz
C0002  C0002  C  0  inf  0  C  2
C0003  C0003  C  0  inf  0  C  2
C0004  C0004  C  0  inf  0  C  2
C0005  C0005  C  0  inf  0  C  2
C0006  C0006  C  0  inf  0  C  2
<VarFrame key=('var',) · total not computed>
var  name  kind  lb  ub  obj  type  nnz
C0202  C0202  I  0  1  0  B  4
C0203  C0203  I  0  1  0  B  4
C0204  C0204  I  0  1  0  B  

## 3. 打开代表行，并检查 family 中被签名隐藏的数值变化

`m.row_families["F0"]` 直接访问加载时生成的行族，族表已确定深度，无需写后缀；它的 `.rows` 是
`m.rows["F0"]` 对应的成员集合。`m.var_families` 是与行族匹配的 depth 1 变量族表。

公式始终限项展示；省略号不表示剩余系数相同。原始 F0 行的 V2 系数是变化的，
而 F1 示例是单位系数。F0 的 params 可给出整个 family 的精确系数分布：
规范方向的 +V2 系数有 190000 个不同值。这与 family 成员共享结构角色并不矛盾。

In [4]:
for label in ("F0", "F1", "F2"):
    family = m.structure.signatures().family(label, depth=2)
    row = family.rows.iloc[0]
    print(family.id, row.name)
    print(row.formula(order="index", max_terms=8))

F0_family = m.structure.signatures().family("F0", depth=2)
display(F0_family.inspect(samples=0))
assert F0_family.params.loc["+V2"].varying


F0 R0001
C0001 - 316.3282564 C0202 - 79.69113535 C0203 - 194.3139522 C0204 - 472.7501309 C0205 - 241.5482321 C0206 - 307.3780612 C0207 - 108.371678 C0208 + ... >= 0
[8 of 201 terms shown]
F1 R0951
C0202 + C0203 + C0204 + C0205 + C0206 + C0207 + C0208 + C0209 + ... = 1
[8 of 200 terms shown]
F2 R2101
-C0002 + 417.27622237 C0202 + 617.76066551 C0402 + 984.39175199 C0602 + 778.56501203 C0802 + 592.55886806 C1002 + 869.61563529 C1202 + 178.1402782 C1402 + ... = 0
[8 of 951 terms shown]
<ConstraintFamily F0: 950 rows, depth=2, mode=template>
  structure: -V0 ×1 | +V2 ×200
  parent: F0
  sign_symmetric: False
  Use .params or .inspect(); .inspect(verbose=True) for diagnostics.
Numerical basis: canonical (original coefficients remain accessible on member edges)
Varying coefficient roles: +V2
<Parameters F0: 950 members; canonical magnitude; roles @ depth=1>
  -V0: coefficient=constant(-1); multiplicity=constant(1)
  +V2: coefficient=varying[30.00170431,499.9971698], distinct=190000; multiplic

## 4. F0 与 F1 是否共享同一批 V2？

抽取 10 条 F0 行，通过限制在 V2 的变量关联找 F1 候选。若一条 F1
包含全部 200 个 V2，比较双方 support：预期共同 200 个变量，仅 F0
多一个 V0。这里检验的是 support 对齐，不是系数相等。

In [5]:
for row in F0.sample(10, random_state=42):
    a = row.vars
    a_v2 = a & V2
    counts = m.matrix.nnz(rows=F1, vars=a_v2, axis="row")
    matches = counts.where(col("nnz") == len(a_v2)).refs("row")
    if len(matches) == 1:
        b = matches.one().vars
        print(row.name, "→", matches.one().name,
              "V2:", len(a_v2),
              "共同/仅F0/仅F1:", len(a & b), len(a - b), len(b - a))
    else:
        print(row.name, "→", len(matches), "个候选", "V2:", len(a_v2))


R0080 → R1030 V2: 200 共同/仅F0/仅F1: 200 1 0
R0100 → R1050 V2: 200 共同/仅F0/仅F1: 200 1 0
R0282 → R1232 V2: 200 共同/仅F0/仅F1: 200 1 0
R0304 → R1254 V2: 200 共同/仅F0/仅F1: 200 1 0
R0387 → R1337 V2: 200 共同/仅F0/仅F1: 200 1 0
R0400 → R1350 V2: 200 共同/仅F0/仅F1: 200 1 0
R0417 → R1367 V2: 200 共同/仅F0/仅F1: 200 1 0
R0519 → R1469 V2: 200 共同/仅F0/仅F1: 200 1 0
R0857 → R1807 V2: 200 共同/仅F0/仅F1: 200 1 0
R0934 → R1884 V2: 200 共同/仅F0/仅F1: 200 1 0


## 5. 给一个 V2 变量恢复两个坐标

变量的 rows 视图给出相邻原始行、family 和精确系数。若它恰好落在一条 F0
和一条 F2，可暂记为 x_(i,j)。先看一个变量，再随机检查 100 个。

In [6]:
x = V2.iloc[123]
f0_edge = x.terms.where(row=F0).one()
f2_edge = x.terms.where(row=F2).one()
display(x.rows)
print({"variable": x.name, "i_row": f0_edge.row.name, "j_row": f2_edge.row.name})


<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R0001  R0001  0  inf  201
R0951  R0951  1  1  200
R2024  R2024  -inf  0  951
R2224  R2224  0  0  951
{'variable': 'C0325', 'i_row': 'R0001', 'j_row': 'R2224'}


In [7]:
sample = V2.sample(100, random_state=42)
P_sample = sample.project(i=F0, j=F2, exactly_one=True)
print(f"checked={len(P_sample)}, bad=0")
display(sample.pattern_counts("family"))


checked=100, bad=0
<Frame key=('pattern',) · total not computed>
pattern  count
4  100


## 6. 全量核验 950 × 200 个坐标

抽样不能证明覆盖完整。下面使用公开的批量投影，检查每个 V2 是否各有一个
F0/F1/F2 坐标，再检查 F0×F2 的唯一性与完整覆盖。严格投影发现零匹配或多匹配时
抛出 CardinalityError；通过 exception.report.violations 可以查询全部违规成员和匹配边。

计算仅使用稀疏索引；不读取名称、系数，不建立完整笛卡尔积。函数依赖报告另外核验
F0 与 F1 坐标的对应关系，并注明未覆盖目标数。


In [8]:
P = V2.project(i=F0, k=F1, j=F2, exactly_one=True, memory_mb=128)
report = P.check("product", ("i", "j"))
display(report)
report.require()
assert len(P) == P.nunique(("i", "j")) == len(F0) * len(F2) == 190000

forward = P.check("function", key="i", value="k")
backward = P.check("function", key="k", value="i")
forward.require()
backward.require()
assert all(n == 0 for n in forward.summary["uncovered_targets"].values())
assert all(n == 0 for n in backward.summary["uncovered_targets"].values())
display(forward, backward)

point = P.loc[x.id]
print(point.var.name, "↔", (point.i.id, point.j.id),
      "↔", (point.i.name, point.j.name))


<ValidationReport: passed; complete>
  operation: product
  scope: full current Frame selection
  checked_records: 190000
  columns: ('i', 'j')
  unique_coordinates: 190000
  duplicate_records: 0
  expected_coordinates: 190000
  missing_coordinates: 0
  outside_domain_records: 0
  violations: 0
<ValidationReport: passed; complete>
  operation: function
  scope: full current Frame selection
  checked_records: 190000
  columns: ('i',)
  key: ('i',)
  value: ('k',)
  uncovered_targets: {'i': 0, 'k': 0}
  violations: 0
<ValidationReport: passed; complete>
  operation: function
  scope: full current Frame selection
  checked_records: 190000
  columns: ('k',)
  key: ('k',)
  value: ('i',)
  uncovered_targets: {'k': 0, 'i': 0}
  violations: 0
C0325 ↔ (0, 2223) ↔ ('R0001', 'R2224')


In [9]:
# 精确系数通过原始变量 Ref 关联，检查基数与完整匹配。
A = m.terms(rows=F0, vars=V2).select("var", a=col("coef"))
Q = P.join(A, on="var", cardinality="1:1", require="left")
S = Q.group_by("i").agg(distinct=col("a").nunique(), amin=col("a").min(), amax=col("a").max())
assert len(Q) == 190000 and len(S) == 950
assert S.column("distinct").max() > 1
display(S.head())


<Frame key=('i',) · total not computed>
i  distinct  amin  amax
R0001  200  -4943277831/10000000  -1548239157/50000000
R0002  200  -4973709687/10000000  -1501788129/50000000
R0003  200  -1243179591/2500000  -752549579/25000000
R0004  200  -1241290367/2500000  -193874129/6250000
R0005  200  -4973374689/10000000  -760172761/25000000


## 让 structural signature 自动提出 indexed 候选

前面的投影验证了我们手动选择的 F0/F1/F2。现在提取 read() 时已完成的 indexed 结果：各层 family 和粗分组提出候选，再按原始成员验证。颜色并不直接代表坐标轴；结果可能包含多个有效坐标视图。


In [ ]:
I = m.structure.indexed(memory_mb=128)
display(I)
# 通过公开原始 ID 映射选择覆盖 V2、使用 F0/F2 轴的自动发现结果。
def same_ids(left, right):
    return len(left) == len(right) and not len(left - right)

T = next(t for t in (I.tensor(record.id) for record in I.tensors)
         if same_ids(t.vars, V2)
         and same_ids(t.axes[0], F0)
         and same_ids(t.axes[1], F2))
display(T)
auto_coordinates = T.coordinates
product = auto_coordinates.check("product", ("i", "j"))
product.require()
display(product)
assert len(auto_coordinates) == auto_coordinates.nunique(("i", "j")) == 190000
assert T.shape == [950, 200] and T.complete_product and T.missing_cells == 0
O = next(o for o in (I.operator(record.id) for record in I.operators)
         if o.tensor == T.id and same_ids(o.rows, F0))
O.verify().require()
display(O.formula(), O.kernel.head(), O.bindings.head())
assert len(O.kernel) > 0
# F0 的系数变化保持为精确稀疏核；不会因共享 family 被解释为常数。
assert O.kernel_statistics["min"] != O.kernel_statistics["max"]
display(m.ir)


## 7. V1 是否是可消去的定义变量？

V1 的 200 个成员都以 F2×1、F5×1 的模式出现。打开一个 F2 等式，可见
一个 V1 与 950 个 V2 通过带权系数相连，因此能代数地解出该 V1。
这仍不代表代入后模型会更好；先估计新增非零项。

In [11]:
V1 = m.vars.family("V1@1")
display(V1.pattern_counts("family"))
y = V1.iloc[0]
display(y.rows)
defining_row = y.rows.family(F2).one()
print("defining row:", defining_row.name)
print(defining_row.formula(max_terms=8))
print("other uses:", (y.rows - F2).to_records())


<Frame key=('pattern',) · total not computed>
pattern  count
2  200
<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R2101  R2101  0  0  951
R2301  R2301  -inf  900000000  400
defining row: R2101
999.11186113 C44402 + 999.07687731 C4402 + 995.77672198 C73802 + 992.75724237 C128402 + 992.30282315 C130402 + 991.42513439 C184602 + 986.42988664 C88402 + 985.5929186 C67002 + ... = 0
[8 of 951 terms shown]
other uses: [{'row': Ref(uid='ae4d0503-fc3f-481a-a607-c371ae24fe12', version='f1bd685529f1132b8bd3ecfa25d1c9f57b179024fb4172fd94cc97f153901868', kind='row', id=2300), 'name': 'R2301', 'lb': -inf, 'ub': Fraction(900000000, 1), 'nnz': 400, 'relation': '<=', 'sense': '<=', 'lhs': -inf, 'rhs': Fraction(900000000, 1), 'coef_min': Fraction(5014327131, 5000000), 'coef_max': Fraction(10952430663, 100000)}]


In [12]:
F5 = m.rows.family("F5@2")
f5 = F5.one()
n_v1_in_f5 = len(f5.vars.where(family=V1))
n_v2_in_definitions = m.matrix.nnz(rows=F2, vars=V2, axis="row").column("nnz")
n_v2_already_in_f5 = len(f5.vars.where(family=V2))

print("V1 terms in F5:", n_v1_in_f5)
print("V2 terms per F2 definition:", n_v2_in_definitions.value_counts())
print("V2 terms already in F5:", n_v2_already_in_f5)
assert n_v1_in_f5 == len(V1)
assert n_v2_in_definitions.min() == n_v2_in_definitions.max() == 950
assert n_v2_already_in_f5 == 0

projected_f5_nnz = f5.nnz - n_v1_in_f5 + n_v2_in_definitions.sum()
print("F5 nnz before / after full substitution:", f5.nnz, projected_f5_nnz)
assert projected_f5_nnz == 190200


V1 terms in F5: 200
V2 terms per F2 definition: <Frame key=('nnz',) · total not computed>
nnz  count
950  200
V2 terms already in F5: 0
F5 nnz before / after full substitution: 400 190200


F5 原有 400 个非零项，其中 200 个是 V1。若全部 V1 代入，
200 条 F2 定义式各带来 950 个不同 V2，F5 将达到 190200 个非零项。
这个估计利用了前面验证的 F2 坐标唯一性以及 F5 原本不含 V2；
它是改写成本的结构证据，不是自动执行 substitution 的建议。

## 8. 查看 198 + 2 的边界分裂

V3 有 198 个成员：196 个关联 F3/F5，另 2 个还关联 F6。
V4 只有 2 个，均关联 F4/F5/F7。F4、F6、F7 的成员数分别为 2、1、1，
因此默认显示即可列出全部成员；长公式仍显式限项。

In [13]:
V3 = m.vars.family("V3@1")
V4 = m.vars.family("V4@1")
display(V3.head())
display(V4)
display(V3.pattern_counts("family"))
display(V4.pattern_counts("family"))

for variable in V4:
    print(variable["name"], {key: variable[key]
                            for key in ("type", "lb", "ub", "obj", "nnz")})
    display(variable.rows)

for label in ("F4", "F6", "F7"):
    family = m.rows.family(label + "@2")
    print(label)
    display(family)
    for row in family:
        print(row.name, row.formula(max_terms=8))

<VarFrame key=('var',) · total not computed>
var  name  kind  lb  ub  obj  type  nnz
C190202  C190202  I  0  1  0  B  3
C190203  C190203  I  0  1  0  B  3
C190206  C190206  I  0  1  0  B  2
C190207  C190207  I  0  1  0  B  2
C190208  C190208  I  0  1  0  B  2
<VarFrame key=('var',) · total not computed>
var  name  kind  lb  ub  obj  type  nnz
C190204  C190204  I  0  1  0  B  3
C190205  C190205  I  0  1  0  B  3
<Frame key=('pattern',) · total not computed>
pattern  count
2  196
3  2
<Frame key=('pattern',) · total not computed>
pattern  count
3  2
C190204 {'type': 'B', 'lb': Fraction(0, 1), 'ub': Fraction(1, 1), 'obj': Fraction(0, 1), 'nnz': 3}
<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R1903  R1903  -inf  0  951
R2301  R2301  -inf  900000000  400
R2303  R2303  2  inf  3
C190205 {'type': 'B', 'lb': Fraction(0, 1), 'ub': Fraction(1, 1), 'obj': Fraction(0, 1), 'nnz': 3}
<RowFrame key=('row',) · total not computed>
row  name  lb  ub  nnz
R1904  R1904  -inf  0  951

## 9. 最后才检查目标变量 V0

V0 的目标系数为 1，只参加 950 条 F0。F1 的 RHS 全为 1，但这不能推出
目标变量等于最大的 F1 RHS：F0 对 V2 使用变化的权重，F1 对它们使用单位系数。
必须先核对同一变量在两组约束中的真实系数，再提出 reformulation。

In [14]:
V0 = m.vars.family("V0@1")
z = V0.one()
print("objective variable:", z.name, "obj =", z.obj)
display(z.rows.value_counts("family"))

rhs = F1.column("ub")
max_rhs = rhs.max()
maximizers = rhs.argmax()
print("F1 RHS distribution:", rhs.value_counts())
print("max RHS:", max_rhs, "attained by", len(maximizers), "rows")
print("first five maximizers:", list(maximizers.head().column("name")))
assert max_rhs == 1 and len(maximizers) == len(F1)

f0_edge = x.terms.where(row=F0).one()
f1_edge = x.terms.where(row=F1).one()
print(x.name, "coef in F0:", f0_edge.coef, "coef in F1:", f1_edge.coef)
f0_row, f1_row = f0_edge.row, f1_edge.row
display(f0_row.vars.where(family=V0))
print(f0_row.name, f0_row.formula(max_terms=6))
print(f1_row.name, f1_row.formula(max_terms=6))


objective variable: C0001 obj = 1
<Frame key=('family',) · total not computed>
family  count
F0@2  950
F1 RHS distribution: <Frame key=('ub',) · total not computed>
ub  count
1  950
max RHS: 1 attained by 950 rows
first five maximizers: ['R0951', 'R0952', 'R0953', 'R0954', 'R0955']
C0325 coef in F0: -846652659/2500000 coef in F1: 1
<VarFrame key=('var',) · total not computed>
var  name  kind  lb  ub  obj  type  nnz
C0001  C0001  C  -inf  inf  1  C  950
R0001 -494.3277831 C0382 - 493.0938692 C0356 - 490.148738 C0279 - 483.5758819 C0241 - 478.567477 C0229 - 475.8665355 C0275 + ... >= 0
[6 of 201 terms shown]
R0951 C0202 + C0203 + C0204 + C0205 + C0206 + C0207 + ... = 1
[6 of 200 terms shown]


## 已验证的事实与下一步

- V2 与 F0×F2 构成完整的一对一坐标：190000 个变量、190000 个不同坐标。
- F0 与 F1 的 V2 support 通过全量坐标对应核验；F0 的系数存在显著数值变化。
- 200 个 V1 分别进入一条 F2 定义式和同一条 F5；全量代入会显著增加 F5 的非零项。
- V3/V4 的少量边界角色可以从邻接行直接解释；V0 的简化仍需严谨的系数推导。

下面关闭模型，释放稀疏数组、SQLite 连接和缓存。如果还要继续交互探索，
请等到最后再运行这一格。

In [15]:
m.close()